<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_1/BPE_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция: Byte Pair Encoding (BPE) — классический алгоритм субсловной токенизации

## 1. Введение

Одной из фундаментальных проблем обработки естественного языка является представление текста в виде последовательности дискретных единиц — токенов. Традиционные подходы, такие как словарная токенизация (разбиение на слова), страдают от проблемы неограниченного словаря и неспособности обрабатывать редкие или неизвестные слова. Альтернативный подход — посимвольная токенизация — решает проблему неизвестных слов, но приводит к слишком длинным последовательностям и теряет морфологическую структуру. Компромиссным решением являются **субсловные (подсловные) методы токенизации**, которые разбивают слова на более мелкие значимые единицы. Среди них наиболее известен **Byte Pair Encoding (BPE)**, первоначально предложенный как алгоритм сжатия данных (Gage, 1994) и впоследствии адаптированный для сегментации текста в нейронном машинном переводе (Sennrich et al., 2016).

В этой лекции мы подробно рассмотрим **классическую версию BPE**, которая работает на уровне символов и итеративно объединяет наиболее частые пары соседних символов или подслов. Мы изучим алгоритм, приведём формальное описание, разберём пошаговый пример, обсудим свойства метода и покажем, как применять обученный BPE к новым словам.

## 2. Идея алгоритма

Основная идея BPE заключается в том, чтобы начать с минимально возможного словаря, состоящего из всех уникальных символов корпуса, и затем **итеративно расширять словарь**, добавляя в него новые токены, образованные слиянием двух наиболее часто встречающихся соседних токенов в текущем представлении корпуса. Таким образом, часто повторяющиеся последовательности символов постепенно становятся едиными токенами, что позволяет эффективно кодировать как частые, так и редкие слова.

Алгоритм является **жадным**: на каждом шаге выбирается пара с максимальной частотой, независимо от того, приведёт ли это в итоге к оптимальному словарю. Несмотря на жадность, на практике BPE даёт хорошие результаты и широко используется в современных моделях (GPT, BERT и др.).

## 3. Формальное описание алгоритма

Пусть задан корпус текста, который после предварительной обработки (например, добавления символа конца слова) представлен в виде одной длинной последовательности символов. Обозначим эту последовательность как

$$ C = (t_1, t_2, \dots, t_n), $$

где $t_i$ — отдельный символ из начального множества $\Sigma$. Начальный словарь токенов $V^{(0)} = \Sigma$, т.е. множество всех уникальных символов, встречающихся в $C$.

BPE выполняет заданное число итераций $K$ (или до достижения желаемого размера словаря). На итерации $k$ (начиная с $k=1$) выполняются следующие шаги:

1. **Подсчёт частот биграмм**. Для каждой упорядоченной пары токенов $(a, b)$, где $a, b \in V^{(k-1)}$ и которые встречаются как соседние в текущей последовательности $C^{(k-1)}$, вычисляется частота:

   $$ f(a,b) = \left| \{ i \in \{1, \dots, n_{k-1} - 1\} : t_i^{(k-1)} = a \text{ и } t_{i+1}^{(k-1)} = b \} \right|. $$

   Здесь $n_{k-1}$ — длина текущей последовательности токенов, а $t_i^{(k-1)}$ — её элементы.

2. **Выбор самой частой пары**. Находится пара $(a^*, b^*)$ такая, что

   $$ (a^*, b^*) = \arg\max_{a,b \in V^{(k-1)}} f(a,b). $$

   Если максимум достигается для нескольких пар, используется **детерминированное правило**: выбирается пара, которая **первой встречается при сканировании корпуса слева направо**. Это правило гарантирует воспроизводимость алгоритма и совпадает с оригинальной реализацией (Sennrich et al., 2016).

3. **Создание нового токена**. Формируется новый токен $c$ путём конкатенации строковых представлений $a^*$ и $b^*$:

   $$ c = a^* \oplus b^*, $$

   где $\oplus$ обозначает операцию конкатенации строк (например, если $a^* = \text{"lo"}$, $b^* = \text{"w"}$, то $c = \text{"low"}$).

4. **Добавление токена в словарь**. Новый токен добавляется в словарь:

   $$ V^{(k)} = V^{(k-1)} \cup \{ c \}. $$

   Это ключевой момент: каждая итерация расширяет словарь ровно на один новый токен — результат слияния самой частотной пары.

5. **Обновление корпуса**. В текущей последовательности $C^{(k-1)}$ все вхождения пары $(a^*, b^*)$ заменяются на один токен $c$. Формально, если в $C^{(k-1)}$ есть подпоследовательность $\dots, t_i, t_{i+1}, \dots$, где $t_i = a^*$ и $t_{i+1} = b^*$, то эти два токена объединяются в один $c$. Процесс повторяется для всех таких соседних пар. Полученная новая последовательность обозначается $C^{(k)}$.

После $K$ итераций алгоритм завершается. Итоговый словарь $V^{(K)}$ содержит исходные символы и $K$ добавленных подслов.

Критерием остановки может быть:
- заданное количество итераций (число слияний);
- достижение желаемого размера словаря;
- отсутствие пар с частотой выше 1 (в этом случае дальнейшее слияние не уменьшит длину корпуса).

**Псевдокод алгоритма**:

```
функция train_bpe(corpus, num_merges):
    vocab = множество всех символов корпуса
    # корпус представлен как список токенов (символов)
    for i in 1..num_merges:
        # подсчёт частот всех соседних пар в текущем корпусе
        pairs = {}
        for j in 1..len(corpus)-1:
            pair = (corpus[j], corpus[j+1])
            pairs[pair] = pairs.get(pair, 0) + 1
        if pairs пусто:
            break
        # выбор пары с максимальной частотой (первая встреченная при равенстве)
        best_pair = максимум по pairs с учётом правила
        new_token = best_pair[0] + best_pair[1]
        vocab.add(new_token)
        # обновление корпуса: замена всех вхождений best_pair на new_token
        новый_corpus = []
        j = 0
        while j < len(corpus):
            if j < len(corpus)-1 and (corpus[j], corpus[j+1]) == best_pair:
                новый_corpus.append(new_token)
                j += 2
            else:
                новый_corpus.append(corpus[j])
                j += 1
        corpus = новый_corpus
    return vocab, список_правил_слияний  # список best_pair в порядке добавления
```

## 4. Пример пошагового выполнения классического BPE

Рассмотрим учебный корпус, состоящий из трёх слов, записанных слитно (без пробелов):
$$ \text{"low lower lowest"}. $$
Для простоты изложения пробелы не учитываются, поэтому весь корпус представляет собой одну строку:
$$ \text{"lowlowerlowest"}. $$
**Важное примечание:** в реальном BPE перед применением алгоритма к каждому слову добавляется специальный символ конца слова (например, `</w>`), чтобы алгоритм не сливал символы, принадлежащие разным словам. В данном учебном примере мы опускаем этот символ, чтобы сосредоточиться на механике слияний; на практике он обязателен.

Этот пример иллюстрирует основные шаги алгоритма и влияние жадного выбора.

### Шаг 0. Инициализация

Начальный словарь — все уникальные символы корпуса:
$$ V^{(0)} = \{ \text{l}, \text{o}, \text{w}, \text{e}, \text{r}, \text{s}, \text{t} \}. $$
Разбиваем корпус на отдельные символы (токены):
$$ C^{(0)} = \text{l o w l o w e r l o w e s t}. $$
Длина последовательности $n_0 = 13$.

### Итерация 1

Подсчитываем частоты всех соседних пар токенов (биграмм) в $C^{(0)}$:

| Пара  | Частота $f$ |
|-------|-----------------|
| (l,o) | 3               |
| (o,w) | 3               |
| (w,l) | 1               |
| (w,e) | 1               |
| (e,r) | 1               |
| (r,l) | 1               |
| (e,s) | 1               |
| (s,t) | 1               |

(Обратите внимание: пары типа (w,l) возникают на стыках слов: `low` + `lower` даёт `...w l...`, и `lower` + `lowest` даёт `...r l...`.)

Максимальная частота равна 3, и она достигается для пар (l,o) и (o,w). Согласно правилу выбора первой встреченной пары при сканировании слева направо, выбираем **(l,o)**. (Если бы выбрали (o,w), результат мог бы немного отличаться.)

Создаём новый токен:
$$ c = \text{"l"} \oplus \text{"o"} = \text{"lo"}. $$
Добавляем его в словарь:
$$ V^{(1)} = \{ \text{l}, \text{o}, \text{w}, \text{e}, \text{r}, \text{s}, \text{t}, \text{lo} \}. $$
Заменяем в $C^{(0)}$ каждую пару (l,o) на токен `lo`. Получаем новую последовательность:
$$ C^{(1)} = \text{lo w lo w e r lo w e s t}. $$
Количество токенов уменьшилось: $n_1 = 10$.

### Итерация 2

Подсчитываем частоты соседних пар в $C^{(1)}$, где токены — это элементы словаря $V^{(1)}$. В последовательности используются токены: `lo`, `w`, `e`, `r`, `s`, `t`.

| Пара     | Частота |
|----------|---------|
| (lo, w)  | 3       |
| (w, e)   | 2       |
| (e, r)   | 1       |
| (r, lo)  | 1       |
| (e, s)   | 1       |
| (s, t)   | 1       |

Наибольшая частота у пары **(lo, w)** — 3. Создаём новый токен:
$$ \text{"lo"} \oplus \text{"w"} = \text{"low"}. $$
Добавляем в словарь:
$$ V^{(2)} = V^{(1)} \cup \{ \text{low} \} = \{ \text{l}, \text{o}, \text{w}, \text{e}, \text{r}, \text{s}, \text{t}, \text{lo}, \text{low} \}. $$
Обновляем корпус, заменяя все пары (lo, w) на `low`:
$$ C^{(2)} = \text{low low e r low e s t}. $$
Теперь последовательность содержит 7 токенов.

### Итерация 3

Анализируем $C^{(2)}$. Токены: `low`, `e`, `r`, `s`, `t`. Соседние пары:

| Пара      | Частота |
|-----------|---------|
| (low, e)  | 2       |
| (e, r)    | 1       |
| (r, low)  | 1       |
| (e, s)    | 1       |
| (s, t)    | 1       |

Максимум у пары **(low, e)** с частотой 2. Образуем токен:
$$ \text{"low"} \oplus \text{"e"} = \text{"lowe"}. $$
Словарь:
$$ V^{(3)} = V^{(2)} \cup \{ \text{lowe} \} = \{ \text{l}, \text{o}, \text{w}, \text{e}, \text{r}, \text{s}, \text{t}, \text{lo}, \text{low}, \text{lowe} \}. $$
Корпус после замены:
$$ C^{(3)} = \text{lowe r lowe s t}. $$
Токенов стало 5.

### Итерация 4

В $C^{(3)}$ токены: `lowe`, `r`, `s`, `t`. Возможные пары:

| Пара       | Частота |
|------------|---------|
| (lowe, r)  | 1       |
| (r, lowe)  | 1       |
| (lowe, s)  | 1       |
| (s, t)     | 1       |

Все частоты равны 1. Согласно правилу выбора первой встреченной пары при сканировании слева направо, выбираем пару **(lowe, r)**, так как она встречается раньше остальных. Создаём токен `lower`:
$$ \text{"lowe"} \oplus \text{"r"} = \text{"lower"}. $$
Словарь:
$$ V^{(4)} = V^{(3)} \cup \{ \text{lower} \} = \{ \text{l}, \text{o}, \text{w}, \text{e}, \text{r}, \text{s}, \text{t}, \text{lo}, \text{low}, \text{lowe}, \text{lower} \}. $$
Обновлённый корпус:
$$ C^{(4)} = \text{lower lowe s t}. $$
Теперь 4 токена.

После четырёх итераций итоговый словарь содержит 11 токенов (7 исходных символов + 4 добавленных). Заметим, что алгоритм последовательно строил цепочку `l` → `lo` → `low` → `lowe`, потому что соответствующие пары были наиболее частотными на каждом этапе. На четвёртой итерации, когда все частоты сравнялись, была выбрана первая встреченная пара `(lowe, r)`, что привело к образованию осмысленного подслова `lower`.

## 5. Применение BPE к новым словам

После того как BPE обучен на корпусе, у нас есть итоговый словарь и **упорядоченный список правил слияния** — пар токенов, которые были объединены на каждой итерации. Этот список позволяет сегментировать любое новое слово, даже если оно не встречалось в обучающем корпусе.

Процедура сегментации нового слова:

1. Разбить слово на отдельные символы (и, при необходимости, добавить символ конца слова `</w>` в конец).
2. Пройти по списку правил слияния **в том порядке, в котором они были добавлены** (от первого к последнему).
3. Для каждого правила `(a, b) -> ab`:
   - Если в текущей последовательности токенов есть соседняя пара `(a, b)`, заменить её на токен `ab`.
   - Повторять замену, пока возможно (для данного правила).
4. Перейти к следующему правилу.
5. Если какой-то символ из входного слова отсутствует в начальном словаре, его обычно заменяют специальным токеном `<unk>` (или используют другую стратегию обработки неизвестных символов, например, байтовое представление).

**Пример**. Предположим, что после обучения на нашем учебном корпусе мы получили следующий список правил (в порядке добавления):
1. `(l, o) -> lo`
2. `(lo, w) -> low`
3. `(low, e) -> lowe`
4. `(lowe, r) -> lower`

Применим эти правила к новому слову `lowest` (которое уже встречалось в корпусе, но для иллюстрации):
- Начальное разбиение: `l o w e s t`.
- Правило 1: заменяем `(l, o)` на `lo` → `lo w e s t`.
- Правило 2: заменяем `(lo, w)` на `low` → `low e s t`.
- Правило 3: заменяем `(low, e)` на `lowe` → `lowe s t`.
- Правило 4: пара `(lowe, r)` отсутствует, пропускаем.

Итоговая сегментация: `lowe s t` (токены `lowe`, `s`, `t`).

Для слова `lower`:
- `l o w e r` → после правил 1–3: `lowe r` → после правила 4: `lower`. Итог: один токен `lower`.

Таким образом, даже если слово отсутствовало в корпусе, оно может быть разбито на известные подслова, что снижает количество неизвестных токенов.

## 6. Свойства и замечания

1. **Жадность и неоптимальность.** BPE на каждом шаге выбирает локально наилучшую пару, что не гарантирует глобально оптимальное разбиение. В примере мы могли бы на первой итерации выбрать пару (o,w), что привело бы к другой последовательности слияний, но итоговый словарь всё равно покрыл бы основные повторяющиеся подслова.

2. **Учёт только соседних пар.** Алгоритм рассматривает исключительно биграммы, т.е. пары непосредственно соседних токенов в текущей последовательности. Токены, которые больше не встречаются в корпусе (как `lo` после того, как все `lo` слились с `w`), не могут образовывать новые пары, даже если они остаются в словаре.

3. **Детерминированность.** При наличии нескольких пар с одинаковой максимальной частотой выбор определяется правилом (первая встреченная при сканировании корпуса). Это делает алгоритм воспроизводимым.

4. **Влияние размера словаря.** Чем больше итераций выполняется, тем крупнее становятся токены и тем меньше длина закодированного текста, но при этом растёт словарь. На практике число слияний (или размер словаря) подбирается эмпирически. Слишком маленький словарь приводит к длинным последовательностям (близко к посимвольной токенизации), слишком большой — к редким токенам и переобучению. Типичные размеры словаря для английского языка в современных моделях составляют от 30 000 до 50 000 токенов.

5. **Обработка неизвестных слов.** После обучения BPE на корпусе, любое новое слово может быть сегментировано с помощью выученных правил слияния (см. раздел 5). Если какой-то символ отсутствует в начальном словаре, он обычно заменяется специальным токеном `<unk>`.

6. **Вычислительная сложность.** Наивная реализация BPE требует на каждой итерации пересчёта частот всех соседних пар, что занимает O(N) для последовательности длины N. При K итерациях общая сложность составляет O(N·K). Для больших корпусов это может быть медленно, поэтому используются оптимизации, например, предварительное вычисление частот слов и применение слияний не ко всему корпусу, а к списку слов с весами. Существуют реализации с очередью с приоритетом, работающие за O(N log N).

7. **Символ конца слова.** Важной деталью практического BPE является добавление специального символа конца слова (например, `</w>`) к каждому слову перед обучением. Это предотвращает слияние символов через границы слов, что критически важно для корректного выделения осмысленных подслов.

8. **Современные модификации.** Классический BPE послужил основой для нескольких широко используемых расширений:
   - **Byte-level BPE** (используется в GPT-2 и последующих моделях) работает на уровне байтов, что позволяет обрабатывать любой текст без предварительной нормализации и поддерживать все символы Unicode.
   - **WordPiece** (используется в BERT) похож на BPE, но при выборе пары максимизирует прирост правдоподобия языковой модели, а не просто частоту.
   - **Unigram Language Model** (используется в SentencePiece) строит словарь вероятностным методом, оптимизируя вероятность корпуса.
   - **SentencePiece** — библиотека, реализующая несколько субсловных алгоритмов и не требующая предварительной токенизации на слова, что удобно для языков без явных пробелов.

## 7. Заключение

Классический BPE — элегантный и эффективный метод построения субсловного словаря, основанный на итеративном слиянии наиболее частых пар токенов. Алгоритм прост в реализации, детерминирован и хорошо масштабируется на большие корпуса. Его главная идея — постепенное укрупнение часто встречающихся последовательностей символов — позволяет уловить как морфемные, так и чисто статистические закономерности языка.

Мы также рассмотрели, как применять обученный BPE к новым словам, и обсудили важные практические аспекты: выбор размера словаря, вычислительную сложность, использование символа конца слова и современные модификации. В следующей части лекции мы подробно остановимся на вероятностных субсловных методах — WordPiece и Unigram Language Model, которые добавляют теоретическую основу и в ряде случаев улучшают качество сегментации.

In [ ]:
from collections import defaultdict, Counter
import re

def train_bpe(corpus, num_merges, verbose=True):
    """
    Классический BPE (Byte Pair Encoding).

    Аргументы:
        corpus: строка (текст) для обучения.
        num_merges: количество итераций слияния.
        verbose: печатать ли детали на каждом шаге.

    Возвращает:
        vocab: множество всех токенов (символы + созданные подслова).
        merges: список правил слияния в порядке добавления (каждый элемент — кортеж (a, b)).
    """
    # 1. Начальный словарь – все уникальные символы корпуса
    vocab = set(corpus)
    # 2. Начальное представление корпуса – список отдельных символов
    #    ВАЖНО: в реальном BPE добавляют символ конца слова </w>, но для простоты опустим.
    corpus_tokens = list(corpus)

    merges = []   # список правил (a, b) в порядке добавления
    current = corpus_tokens[:]

    if verbose:
        print("="*60)
        print("Начальное состояние:")
        print(f"Словарь: {sorted(vocab)}")
        print(f"Корпус (токены): {' '.join(current)}")
        print()

    for i in range(1, num_merges+1):
        # Шаг 1: подсчёт частот всех соседних пар
        pair_counts = defaultdict(int)
        # Также запоминаем порядок первого появления каждой пары (для разрешения конфликтов)
        first_seen = {}
        for j in range(len(current)-1):
            pair = (current[j], current[j+1])
            if pair not in first_seen:
                first_seen[pair] = j
            pair_counts[pair] += 1

        if not pair_counts:
            if verbose:
                print("Нет соседних пар – остановка.")
            break

        # Шаг 2: выбор самой частой пары
        # Сначала находим максимальную частоту
        max_freq = max(pair_counts.values())
        # Собираем все пары с этой частотой
        best_candidates = [(p, first_seen[p]) for p, f in pair_counts.items() if f == max_freq]
        # Сортируем по индексу первого появления (выбираем самую левую)
        best_candidates.sort(key=lambda x: x[1])
        best_pair = best_candidates[0][0]

        # Шаг 3: создание нового токена
        new_token = best_pair[0] + best_pair[1]
        vocab.add(new_token)
        merges.append(best_pair)

        # Шаг 4: обновление корпуса – замена всех вхождений best_pair на new_token
        new_current = []
        j = 0
        while j < len(current):
            if j < len(current)-1 and (current[j], current[j+1]) == best_pair:
                new_current.append(new_token)
                j += 2
            else:
                new_current.append(current[j])
                j += 1
        current = new_current

        if verbose:
            print(f"--- Итерация {i} ---")
            print(f"Частоты пар (топ-5): {sorted(pair_counts.items(), key=lambda x: -x[1])[:5]}")
            print(f"Выбрана пара: {best_pair}  (частота {max_freq})")
            print(f"Новый токен: '{new_token}'")
            print(f"Обновлённый словарь: {sorted(vocab)}")
            print(f"Обновлённый корпус: {' '.join(current)}")
            print()

    return vocab, merges, current

def apply_bpe(word, merges):
    """
    Применить обученные правила слияния к новому слову.

    Аргументы:
        word: строка (слово) для сегментации.
        merges: список правил слияния в порядке добавления (как из train_bpe).

    Возвращает:
        список токенов после применения всех правил.
    """
    tokens = list(word)
    for a, b in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == (a, b):
                new_tokens.append(a+b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


# Пример из лекции (без пробелов)
corpus = "lowlowerlowest"
num_merges = 4

vocab, merges, final_tokens = train_bpe(corpus, num_merges, verbose=True)

print("="*60)
print("ИТОГОВЫЙ РЕЗУЛЬТАТ:")
print(f"Словарь: {sorted(vocab)}")
print(f"Правила слияния (в порядке добавления): {merges}")
print(f"Финальный корпус: {' '.join(final_tokens)}")

# Byte-level BPE — современный алгоритм субсловной токенизации

## 1. Введение

Классический Byte Pair Encoding (BPE), рассмотренный в предыдущей лекции, работает на уровне символов: алфавит состоит из всех уникальных символов, встречающихся в обучающем корпусе. Такой подход хорошо зарекомендовал себя в задачах машинного перевода и ранних языковых моделях, однако у него есть ряд ограничений, которые становятся критичными при переходе к многоязычным и открытым системам.

**Проблемы классического BPE:**

- **Зависимость от предварительной токенизации.** Классический BPE обычно применяется после разбиения текста на слова (например, по пробелам) и добавления специального символа конца слова `</w>`. Это делает алгоритм зависимым от правил сегментации конкретного языка. Для языков без явных пробелов (китайский, японский, тайский) такая токенизация нетривиальна.
- **Обработка неизвестных символов.** Если на этапе инференса встречается символ, отсутствовавший в обучающем корпусе (редкий эмодзи, символ другого языка, математический знак), стандартный подход — заменить его на специальный токен `<unk>`. Такая замена теряет информацию и может ухудшить качество генерации или понимания.
- **Большой начальный словарь.** Размер алфавита Unicode огромен (более 140 000 символов). Даже после фильтрации редких символов начальный словарь может содержать десятки тысяч элементов, что усложняет обучение и увеличивает размер модели.

**Решение: переход на уровень байтов**

Современным развитием BPE является **Byte-level BPE (BBPE)**, предложенный в работе «Language Models are Unsupervised Multitask Learners» (Radford et al., 2019) и использованный в моделях GPT-2 и GPT-3. Вместо символов в качестве базовых единиц используются **байты**. Каждый символ Unicode представляется последовательностью байтов (обычно в кодировке UTF-8), и алгоритм BPE выполняется на уровне байтов, а не символов.

**Преимущества Byte-level BPE:**

- **Универсальность:** базовый алфавит фиксирован и содержит 256 возможных байтов (от 0 до 255). Это позволяет обрабатывать любые тексты без предварительной нормализации или составления словаря символов.
- **Открытость словаря:** даже если символ не встречался в обучении, он всё равно может быть закодирован через последовательность байтов, поскольку все байты известны. Неизвестные символы не теряются.
- **Единообразие:** нет необходимости в сложной предобработке, пробелы и знаки препинания рассматриваются как обычные байты, а границы слов могут выучиваться автоматически.
- **Устойчивость к шуму:** алгоритм естественно обрабатывает опечатки, эмодзи, редкие символы.

В этой лекции мы подробно рассмотрим Byte-level BPE: его мотивацию, алгоритм, отличия от классического BPE, приведём пример и обсудим практические аспекты.

## 2. Идея алгоритма

Byte-level BPE полностью сохраняет основную идею классического BPE, но меняет **базовый уровень представления**: вместо символов текста используются байты, полученные в результате кодирования текста в UTF-8.

Исходный корпус преобразуется в длинную последовательность байтов. Начальный словарь состоит из всех 256 возможных байтов (значения от 0 до 255). Далее алгоритм итеративно объединяет наиболее часто встречающиеся **пары соседних байтов** (или уже укрупнённых токенов) в новые токены, которые представляют собой последовательности байтов. Этот процесс ничем не отличается от классического BPE, за исключением того, что элементарными единицами являются не символы, а байты.

Благодаря этому подходу отпадает необходимость в предварительном определении алфавита символов, а проблема неизвестных символов исчезает: любой символ, даже не встречавшийся в обучении, может быть представлен комбинацией байтов. Кроме того, пробелы и другие управляющие символы становятся обычными байтами, что позволяет алгоритму самостоятельно выучивать осмысленные границы слов.

Алгоритм по-прежнему является **жадным** и **детерминированным** при заданном правиле разрешения неоднозначностей.

## 3. Формальное описание алгоритма

Пусть задан корпус текста, который преобразован в последовательность байтов с использованием кодировки UTF-8. Обозначим множество всех возможных байтов как

$$ \mathcal{B} = \{0, 1, 2, \dots, 255\}. $$

Каждый байт $b_i$ принадлежит $\mathcal{B}$. После преобразования весь корпус представляется как одна длинная последовательность байтов:

$$ C^{(0)} = (b_1, b_2, \dots, b_n), $$

где $n$ — общее количество байтов в корпусе.

Начальный словарь токенов полагается равным всему множеству байтов:

$$ V^{(0)} = \mathcal{B}. $$

Byte-level BPE выполняет заданное число итераций $K$ (число слияний). На итерации $k$ (начиная с $k=1$) выполняются следующие шаги:

1. **Подсчёт частот биграмм.** Для каждой упорядоченной пары токенов $(x, y)$, где $x, y \in V^{(k-1)}$ и которые встречаются как соседние в текущей последовательности $C^{(k-1)}$, вычисляется частота:

   $$ f^{(k)}(x,y) = \left| \{ i \in \{1, \dots, n_{k-1}-1\} : t_i^{(k-1)} = x \text{ и } t_{i+1}^{(k-1)} = y \} \right|. $$

   Здесь $n_{k-1}$ — длина текущей последовательности токенов, а $t_i^{(k-1)}$ — её элементы.

2. **Выбор самой частой пары.** Находится пара $(x^*, y^*)$ такая, что

   $$ (x^*, y^*) = \arg\max_{x,y \in V^{(k-1)}} f^{(k)}(x,y). $$

   Если максимум достигается для нескольких пар, используется детерминированное правило: выбирается пара, которая **первой встречается при сканировании корпуса слева направо**. Это правило гарантирует воспроизводимость.

3. **Создание нового токена.** Формируется новый токен $z$ путём конкатенации байтовых последовательностей $x^*$ и $y^*$. Поскольку $x^*$ и $y^*$ сами являются последовательностями байтов (возможно, уже укрупнёнными), результатом также является последовательность байтов:

   $$ z = x^* \oplus y^*, $$

   где $\oplus$ обозначает операцию конкатенации байтовых последовательностей.

4. **Добавление токена в словарь.** Новый токен добавляется в словарь:

   $$ V^{(k)} = V^{(k-1)} \cup \{ z \}. $$

   Размер словаря увеличивается ровно на один токен на каждой итерации.

5. **Обновление корпуса.** В текущей последовательности $C^{(k-1)}$ все вхождения пары $(x^*, y^*)$ заменяются на один токен $z$. Формально, если в $C^{(k-1)}$ есть подпоследовательность $\dots, t_i, t_{i+1}, \dots$, где $t_i = x^*$ и $t_{i+1} = y^*$, то эти два токена объединяются в один $z$. Процесс повторяется для всех таких соседних пар. Полученная новая последовательность обозначается $C^{(k)}$.

После $K$ итераций алгоритм завершается. Итоговый словарь $V^{(K)}$ содержит исходные 256 байтов и $K$ добавленных байтовых подслов. Обычно $K$ выбирается так, чтобы итоговый размер словаря был порядка $30\,000$–$50\,000$ токенов.

**Критерии остановки** аналогичны классическому BPE:
- заданное количество итераций;
- достижение желаемого размера словаря;
- отсутствие пар с частотой выше 1.

**Псевдокод алгоритма**:

```
функция train_byte_level_bpe(text, num_merges):
    # Преобразуем текст в последовательность байтов (UTF-8)
    byte_sequence = text.encode('utf-8')   # список целых чисел 0..255
    vocab = set(range(256))                # начальный словарь = все байты
    merges = []                            # список правил слияния

    for i in 1..num_merges:
        # Подсчёт частот пар соседних токенов
        pairs = Counter()
        for j in 0..len(byte_sequence)-2:
            pair = (byte_sequence[j], byte_sequence[j+1])
            pairs[pair] += 1

        if not pairs:
            break

        # Выбор самой частой пары (первая встреченная при равенстве)
        best_pair = max(pairs, key=pairs.get)
        # Новый токен — конкатенация байтовых последовательностей
        new_token = best_pair[0] + best_pair[1]  # обе части — байтовые строки/кортежи
        merges.append(best_pair)
        vocab.add(new_token)

        # Обновление последовательности: замена всех вхождений best_pair на new_token
        new_seq = []
        j = 0
        while j < len(byte_sequence):
            if (j < len(byte_sequence)-1 and
                (byte_sequence[j], byte_sequence[j+1]) == best_pair):
                new_seq.append(new_token)
                j += 2
            else:
                new_seq.append(byte_sequence[j])
                j += 1
        byte_sequence = new_seq

    return vocab, merges
```

## 4. Пример пошагового выполнения Byte-level BPE

Рассмотрим учебный корпус, состоящий из трёх английских слов с пробелами:

$$ \text{"low lower lowest"}. $$

В отличие от классического примера, мы **не будем удалять пробелы** — в байтовом BPE пробел является обычным байтом и участвует в слияниях наравне с другими.

### Шаг 0. Представление в байтах

Для простоты предположим, что все символы корпуса принадлежат ASCII, поэтому каждый символ кодируется одним байтом. (Для не-ASCII символов кодировка UTF-8 даёт несколько байтов на символ, но принцип не меняется.) Байтовое представление строки выглядит так (в десятичных значениях):

- 'l' → 108
- 'o' → 111
- 'w' → 119
- пробел → 32
- 'e' → 101
- 'r' → 114
- 's' → 115
- 't' → 116

Вся строка "low lower lowest" в байтах:

$$ C^{(0)} = (108, 111, 119, 32, 108, 111, 119, 101, 114, 32, 108, 111, 119, 101, 115, 116). $$

Для наглядности будем записывать байты соответствующими ASCII-символами (помня, что пробел — это байт 32):

$$ C^{(0)} = \text{l o w \_ l o w e r \_ l o w e s t}. $$

Длина последовательности $n_0 = 16$ байтов.

Начальный словарь $V^{(0)}$ содержит все 256 байтов, но в корпусе встречаются только 8 уникальных байтов: `l`, `o`, `w`, `_` (пробел), `e`, `r`, `s`, `t`.

### Итерация 1

Подсчитываем частоты всех соседних пар байтов в $C^{(0)}$:

| Пара       | Частота $f$ |
|------------|-------------|
| (l, o)     | 3           |
| (o, w)     | 3           |
| (w, _)     | 2           |
| (_, l)     | 2           |
| (w, e)     | 2           |
| (e, r)     | 1           |
| (r, _)     | 1           |
| (e, s)     | 1           |
| (s, t)     | 1           |

Максимальная частота равна 3, она достигается для пар (l, o) и (o, w). По правилу первой встреченной пары выбираем **(l, o)**, так как она встречается раньше.

Создаём новый токен объединением байтов `l` и `o` (последовательность из двух байтов):

$$ z = \text{"l"} \oplus \text{"o"} = \text{"lo"}. $$

Добавляем его в словарь:

$$ V^{(1)} = V^{(0)} \cup \{ \text{"lo"} \}. $$

Обновляем корпус, заменяя все вхождения пары (l, o) на токен `lo`:

$$ C^{(1)} = \text{lo w \_ lo w e r \_ lo w e s t}. $$

Количество токенов уменьшилось: $n_1 = 16 - 3 = 13$.

### Итерация 2

В $C^{(1)}$ токены: `lo`, `w`, `_`, `e`, `r`, `s`, `t`. Подсчитываем частоты соседних пар:

| Пара      | Частота |
|-----------|---------|
| (lo, w)   | 3       |
| (w, _)    | 2       |
| (_, lo)   | 2       |
| (w, e)    | 2       |
| (e, r)    | 1       |
| (r, _)    | 1       |
| (e, s)    | 1       |
| (s, t)    | 1       |

Максимальная частота 3 у пары **(lo, w)**. Создаём новый токен:

$$ \text{"lo"} \oplus \text{"w"} = \text{"low"}. $$

Словарь:

$$ V^{(2)} = V^{(1)} \cup \{ \text{"low"} \}. $$

Обновляем корпус, заменяя все (lo, w) на `low`:

$$ C^{(2)} = \text{low \_ low e r \_ low e s t}. $$

Длина $n_2 = 13 - 3 = 10$.

### Итерация 3

В $C^{(2)}$ токены: `low`, `_`, `e`, `r`, `s`, `t`. Перечислим все соседние пары и их частоты:

| Пара       | Частота |
|------------|---------|
| (low, _)   | 1       |
| (_, low)   | 2       |
| (low, e)   | 2       |
| (e, r)     | 1       |
| (r, _)     | 1       |
| (e, s)     | 1       |
| (s, t)     | 1       |

Теперь две пары имеют частоту 2: (_, low) и (low, e). По правилу первой встреченной пары при сканировании корпуса слева направо первой встречается пара **(_, low)** — она находится между пробелом и вторым словом `low`, тогда как (low, e) появляется позже (между `low` и `e` в слове `lower`). Поэтому выбираем её.

Создаём токен, объединяя пробел и `low`:

$$ \text{"\_"} \oplus \text{"low"} = \text{"\_low"}. $$

Словарь:

$$ V^{(3)} = V^{(2)} \cup \{ \text{"\_low"} \}. $$

Обновляем корпус, заменяя все вхождения (_, low) на `_low`. В $C^{(2)}$ такие пары встречаются дважды: после первого `low` и после `r` перед третьим `low`. Получаем:

$$ C^{(3)} = \text{low \_low e r \_low e s t}. $$

Длина $n_3 = 10 - 2 = 8$.

### Итерация 4

В $C^{(3)}$ токены: `low`, `_low`, `e`, `r`, `s`, `t`. Соседние пары:

| Пара          | Частота |
|---------------|---------|
| (low, _low)   | 1       |
| (_low, e)     | 2       |
| (e, r)        | 1       |
| (r, _low)     | 1       |
| (e, s)        | 1       |
| (s, t)        | 1       |

Максимальная частота 2 у пары **(_low, e)**. Выбираем её.

Создаём токен `_lowe`:

$$ \text{"\_low"} \oplus \text{"e"} = \text{"\_lowe"}. $$

Словарь:

$$ V^{(4)} = V^{(3)} \cup \{ \text{"\_lowe"} \}. $$

Обновляем корпус: заменяем все (_low, e) на `_lowe`. Их два, получаем:

$$ C^{(4)} = \text{low \_lowe r \_lowe s t}. $$

Длина $n_4 = 8 - 2 = 6$.

### Итерация 5

В $C^{(4)}$ токены: `low`, `_lowe`, `r`, `s`, `t`. Соседние пары:

| Пара            | Частота |
|-----------------|---------|
| (low, _lowe)    | 1       |
| (_lowe, r)      | 1       |
| (r, _lowe)      | 1       |
| (_lowe, s)      | 1       |
| (s, t)          | 1       |

Все частоты равны 1. По правилу первой встреченной пары выбираем **(low, _lowe)**, так как она первая. (Если бы мы хотели получить более «полезное» слияние, можно было бы выбрать другую, но правило фиксировано.)

Создаём токен `low_lowe` (конкатенация `low` и `_lowe`):

$$ \text{"low"} \oplus \text{"\_lowe"} = \text{"low\_lowe"}. $$

Словарь:

$$ V^{(5)} = V^{(4)} \cup \{ \text{"low\_lowe"} \}. $$

Обновляем корпус:

$$ C^{(5)} = \text{low\_lowe r \_lowe s t}. $$

Длина $n_5 = 6 - 1 = 5$.

После пяти итераций итоговый словарь содержит 256 байтов + 5 добавленных токенов: `lo`, `low`, `_low`, `_lowe`, `low_lowe`. Заметим, как пробелы естественным образом вошли в состав токенов, образуя осмысленные подслова с пробелами в начале (`_low`, `_lowe`). Это типично для Byte-level BPE: частые слова с пробелами объединяются в единые токены.

## 5. Применение Byte-level BPE к новым словам

После обучения мы имеем итоговый словарь и **упорядоченный список правил слияния** (пар токенов, которые объединялись). Этот список позволяет сегментировать любой новый текст, даже с символами, не встречавшимися в обучении.

Процедура кодирования нового текста:

1. Преобразовать текст в последовательность байтов (UTF-8).
2. Пройти по списку правил слияния **в том порядке, в котором они были добавлены**.
3. Для каждого правила $(x, y) \to z$: если в текущей последовательности есть соседняя пара $(x, y)$, заменить её на токен $z$. Повторять, пока возможно.
4. Если какой-то байт (или токен) отсутствует в словаре (что невозможно, так как все байты изначально в словаре), то он остаётся как есть.
5. Полученная последовательность токенов передаётся модели.

**Пример.** Предположим, после обучения на нашем корпусе мы получили правила в порядке добавления:
1. `(l, o) -> lo`
2. `(lo, w) -> low`
3. `(_, low) -> _low`
4. `(_low, e) -> _lowe`
5. `(low, _lowe) -> low_lowe`

Применим их к новому слову `lowest` (без пробела):
- Исходные байты: `l o w e s t`
- Правило 1: `l o` → `lo`: `lo w e s t`
- Правило 2: `lo w` → `low`: `low e s t`
- Правило 3: пара `(_, low)` отсутствует (нет пробела перед `low`)
- Правило 4: пара `(_low, e)` отсутствует
- Правило 5: пара `(low, _lowe)` отсутствует

Итог: токены `low`, `e`, `s`, `t`.

Для слова `lower`:
- `l o w e r` → после правил 1–2: `low e r` → правило 3 не применяется (нет пробела) → правило 4 не применяется → итог: `low e r`.

Для текста `low lower lowest` (как в обучении) получим сегментацию, близкую к той, что мы получили на этапе обучения: `low _lowe r _lowe s t` (если остановились после 4-й итерации) или `low_lowe r _lowe s t` (после 5-й).

**Обработка неизвестных символов.** Важное преимущество Byte-level BPE проявляется при встрече символов, которых не было в обучении. Например, слово `"привет"` в кодировке UTF-8 состоит из следующих байтов (в шестнадцатеричном виде):
- 'п' → D0 BF
- 'р' → D1 80
- 'и' → D0 B8
- 'в' → D0 B2
- 'е' → D0 B5
- 'т' → D1 82

Если в обучении не было кириллицы, эти байты всё равно присутствуют в начальном словаре (как байты), и алгоритм сможет их обработать. Сегментация будет содержать отдельные байты или их слитые комбинации (если они были частыми в обучении). Никакого `<unk>` для символов не потребуется.

## 6. Свойства и замечания

1. **Универсальность и открытость словаря.** Благодаря тому что начальный словарь содержит все 256 байтов, любой текст может быть закодирован без потери информации. Неизвестные символы не заменяются на `<unk>`, а представляются байтовыми последовательностями.

2. **Пробелы и границы слов.** Пробел является обычным байтом и может участвовать в слияниях. Это приводит к образованию токенов, включающих пробелы (например, `_low`, ` the`). Модель учится автоматически выделять частые словосочетания с пробелами.

3. **Длина последовательностей.** Байтовое представление увеличивает длину последовательностей по сравнению с символьным BPE, особенно для языков с многобайтовыми символами. Однако последующие слияния сокращают длину, и итоговое количество токенов обычно сопоставимо с классическим BPE при том же размере словаря.

4. **Вычислительная сложность.** Алгоритм имеет ту же сложность $O(N \cdot K)$, где $N$ — количество байтов в корпусе, $K$ — число слияний. На практике применяются оптимизации, например, предварительный подсчёт частот байтовых биграмм и использование кучи для быстрого поиска максимума.

5. **Детерминированность.** При фиксированном правиле разрешения равенства частот алгоритм полностью воспроизводим.

6. **Сравнение с классическим BPE.**

| Характеристика          | Классический BPE | Byte-level BPE |
|-------------------------|------------------|----------------|
| Базовые единицы         | Символы Unicode  | Байты UTF-8    |
| Размер начального словаря | Зависит от корпуса | Фиксирован: 256 |
| Обработка неизвестных символов | Замена на `<unk>` | Полное представление через байты |
| Зависимость от языка    | Требует списка символов | Универсален |
| Учёт пробелов           | Часто требует специальной обработки | Пробелы — обычные байты |
| Примеры моделей         | Transformer, ранние NMT | GPT-2, GPT-3, RoBERTa (в некоторых конфигурациях) |

7. **Реализации.** Byte-level BPE реализован в библиотеке `byte-pair-encoding` (оригинальная реализация OpenAI), а также в `transformers` от HuggingFace (класс `GPT2Tokenizer`). При работе с многобайтовыми символами важно учитывать, что один символ может быть разделён между несколькими токенами.

8. **Особенности работы с UTF-8.** UTF-8 — это кодировка переменной длины: символы ASCII занимают 1 байт, многие европейские и кириллические — 2 байта, азиатские иероглифы — 3 байта, эмодзи — 4 байта. Поэтому начальная последовательность может быть значительно длиннее, чем в символьном BPE. Однако BPE быстро выучивает частые байтовые последовательности, соответствующие целым символам или даже группам символов, и длина сокращается до уровня, сопоставимого с символьным подходом. В GPT-2 средняя длина токена составляет около 3.5 байта.

9. **Специальные токены.** В байтовом BPE не требуется добавлять символ конца слова `</w>`, так как пробел уже является байтом и выполняет аналогичную функцию. Однако иногда для удобства декодирования или обозначения границ слов используют специальные токены (например, `Ġ` в некоторых реализациях SentencePiece), но в оригинальном Byte-level BPE OpenAI пробел остаётся обычным байтом.

## 7. Заключение

Byte-level BPE — это современная модификация классического BPE, которая переносит алгоритм на уровень байтов, обеспечивая универсальность, открытость словаря и устойчивость к редким символам. Она лежит в основе таких мощных моделей, как GPT-2 и GPT-3, и стала стандартным подходом для субсловной токенизации в больших языковых моделях.

В этой лекции мы подробно рассмотрели мотивацию, формальное описание, пример работы и практические аспекты Byte-level BPE. В следующей части мы обратимся к вероятностным субсловным методам — WordPiece и Unigram Language Model, которые используют более изощрённые критерии для построения словаря и сегментации.